# Beyond BLAST Notebook

In [23]:
ENV["JULIA_PKG_PRECOMPILE_AUTO"] = 0;
using Pkg
Pkg.activate("blast_code"; io=devnull)
Pkg.resolve(; io=devnull)
Pkg.instantiate(; io=devnull)

using Revise 

# using Base.Threads, NPZ, DataInterpolations, Interpolations, FastChebInterp
# using BenchmarkTools, FFTW, FastTransforms, Dates, TOML, Plots, Plots.Measures
# using QuadGK, LaTeXStrings, Tullio, StaticArrays, LoopVectorization, LinearAlgebra
# using Unitful, SpecialFunctions, DifferentialEquations, Cosmology, NumericalIntegration
# using CSV, DataFrames, JSON, OrderedCollections
;

In [26]:
include("blast_code/src/Blast.jl")
include("blast_code/src/blast_tutorials.jl")
include("blast_code/src/galaxy_galaxy.jl")
include("blast_code/src/shear_shear.jl")
include("blast_code/src/config.jl")
include("blast_code/src/paths.jl")
include("blast_code/src/plot_config.jl")
using .galaxy_galaxy
using .shear_shear
using .Blast
using .blast_tutorials

In [27]:
using SpecialFunctions, QuadGK, DataInterpolations, Base.Threads

In [35]:
paths = setup_output_directories()
grid_data = setup_cosmology_grid() 
grids = Blast.generate_k_grids(grid_data.kmin, grid_data.kmax, grid_data.Nk, grid_data.Nkp, grid_data.Nkpp; sorting=false) 

jl(l, x) = SpecialFunctions.sphericalbesselj(l, x)

# --- W(χ) continuo: interpolo l'array già calcolato da galaxy_prefactor ---
# (è lecito: non sto usando Chebyshev/FFT, sto solo rappresentando i dati discreti di W)
gal_prefact_W = npzread(joinpath("/Users/anvi/Desktop/cosmo/notebooks/out/runs/run_2026_07_13_204211_350k_350kp/galaxy_prefactor_W.npy"))
W_interp = DataInterpolations.AkimaInterpolation(
    gal_prefact_W, grid_data.x, extrapolation = ExtrapolationType.Linear
)
;

Folders in place: out/runs/run_2026_07_13_233127, out/runs/run_2026_07_13_233127/plots and out/runs/run_2026_07_13_233127/quantities


In [32]:
# --- P(k) continuo: riuso l'interpolante log-log già costruito dal notebook ---
# (replica esattamente quello che fa il notebook per restare confrontabile)
Pk_of_k(k) = sqrt(10^InterpPmm(grid_data.z_of_χ(grid_data.xmin), log10(k)) *
                   10^InterpPmm(grid_data.z_of_χ(grid_data.xmax), log10(k)))


Pk_of_k (generic function with 1 method)

In [36]:

# --- integrale interno in χ ---
function Delta_l(l::Real, k::Real, kp::Real, xmin::Real, xmax::Real, Wf; rtol = 1e-8)
    integrand(chi) = Wf(chi) * jl(l, k*chi) * jl(l, kp*chi)
    val, _ = quadgk(integrand, xmin, xmax; rtol = rtol)
    return val
end

# --- integrale esterno in k, per un singolo elemento (k1,k2,l) ---
function S_element(l::Real, k1::Real, k2::Real,
                    kmin::Real, kmax::Real, xmin::Real, xmax::Real,
                    Wf, Pk; rtol_k = 1e-6, rtol_chi = 1e-8)
    integrand(k) = k^2 * Pk(k) *
                   Delta_l(l, k, k1, xmin, xmax, Wf; rtol = rtol_chi) *
                   Delta_l(l, k, k2, xmin, xmax, Wf; rtol = rtol_chi)
    val, _ = quadgk(integrand, kmin, kmax; rtol = rtol_k)
    return (2/π) * val   # = pref_gg * val
end

# --- tensore completo S_lkk_gg, calcolato a mano ---
function S_lkk_gg_bruteforce(ell_list, kp_grid, kpp_grid,
                              kmin, kmax, xmin, xmax, Wf, Pk;
                              rtol_k = 1e-6, rtol_chi = 1e-8)
    Nl, Nkp, Nkpp = length(ell_list), length(kp_grid), length(kpp_grid)
    S = zeros(Float64, Nkp, Nkpp, Nl)
    @threads for il in 1:Nl
        l = ell_list[il]
        for ip in 1:Nkp, ipp in 1:Nkpp
            S[ip, ipp, il] = S_element(l, kp_grid[ip], kpp_grid[ipp],
                                        kmin, kmax, xmin, xmax, Wf, Pk;
                                        rtol_k = rtol_k, rtol_chi = rtol_chi)
        end
    end
    return S
end

S_lkk_gg_bruteforce (generic function with 1 method)

In [37]:
# --- Sottoinsieme di test: pochi ℓ e pochi k1,k2 ---
il_test  = 1:3          # primi 3 multipoli
ik_test  = 1:5          # primi 5 k1
ikp_test = 1:5          # primi 5 k2

ell_test  = grid_data.ℓ[il_test]
kp_test   = grids.kp_grid[ik_test]
kpp_test  = grids.kp_grid[ikp_test]   # stessa griglia, se kp_grid == kpp_grid nel tuo setup

# --- Chiamata alla funzione brute-force ---
t0 = time()
S_test = S_lkk_gg_bruteforce(
    ell_test, kp_test, kpp_test,
    grid_data.kmin, grid_data.kmax,
    grid_data.xmin, grid_data.xmax,
    W_interp, Pk_of_k;
    rtol_k = 1e-6, rtol_chi = 1e-8
)
t1 = time()
println("Tempo impiegato per il sottoinsieme di test: ", round(t1 - t0, digits=2), " s")
println("Dimensioni di S_test: ", size(S_test))

# --- Confronto con il risultato già salvato dalla pipeline veloce ---
S_ref_test = S_lkk_gg[ik_test, ikp_test, il_test]

rel_err = abs.(S_test .- S_ref_test) ./ abs.(S_ref_test)
println("Errore relativo massimo: ", maximum(rel_err))
println("Errore relativo medio:   ", sum(rel_err) / length(rel_err))

LoadError: TaskFailedException

[91m    nested task error: [39mUndefVarError: `InterpPmm` not defined
    Stacktrace:
      [1] [0m[1mPk_of_k[22m[0m[1m([22m[90mk[39m::[0mFloat64[0m[1m)[22m
    [90m    @[39m [32mMain[39m [90m./[39m[90m[4mIn[32]:3[24m[39m
      [2] [0m[1m(::var"#integrand#29"{…})[22m[0m[1m([22m[90mk[39m::[0mFloat64[0m[1m)[22m
    [90m    @[39m [32mMain[39m [90m./[39m[90m[4mIn[36]:13[24m[39m
      [3] [0m[1mevalrule[22m[0m[1m([22m[90mf[39m::[0mvar"#integrand#29"[90m{…}[39m, [90ma[39m::[0mFloat64, [90mb[39m::[0mFloat64, [90mx[39m::[0mVector[90m{…}[39m, [90mw[39m::[0mVector[90m{…}[39m, [90mwg[39m::[0mVector[90m{…}[39m, [90mnrm[39m::[0mtypeof(norm)[0m[1m)[22m
    [90m    @[39m [36mQuadGK[39m [90m~/.julia/packages/QuadGK/5mgi5/src/[39m[90m[4mevalrule.jl:0[24m[39m
      [4] [0m[1m#8[22m
    [90m    @[39m [90m~/.julia/packages/QuadGK/5mgi5/src/[39m[90m[4madapt.jl:54[24m[39m